#LAB 03 — Word Representations and Embeddings

#Experiment 1 — Word-context representation

In [7]:
from cooccurrence import build_vocabulary, build_cooccurrence_matrix, compute_matrix_stats, word_similarity

In [14]:
text_corpus = [
    "the cat eats fish and drinks milk".split(),
    "the dog eats meat and chews bones".split(),
    "the cat likes warm milk".split(),
    "the dog likes fresh meat".split(),
    "a king is a powerful man".split(),
    "a queen is a powerful woman".split(),
    "the doctor treats sick patients in the hospital".split(),
    "the physician heals sick patients in the clinic".split(),
    "apples and bananas are healthy fruits".split(),
    "oranges and lemons are sour fruits".split(),
    "machine learning is a subset of artificial intelligence".split(),
    "natural language processing makes computers understand human text".split(),
    "the quick brown fox jumps over the lazy dog".split(),
    "a smart student studies hard for the exam".split(),
    "the brilliant student reads many books".split(),
    "the fast car races on the track".split(),
    "the quick vehicle drives on the road".split(),
    "the chef cooks delicious food in the kitchen".split(),
    "the baker makes fresh bread in the bakery".split(),
    "water is essential for all living creatures".split()
]

vocab = build_vocabulary(text_corpus)
print(f"Vocab size: {len(vocab)} word")


Vocab size: 89 từ


In [15]:
import pandas as pd

# Khởi tạo danh sách lưu trữ kết quả để so sánh đối chiếu
comparison_results = []

# Vocabulary size và Matrix size là cố định với cùng một tập corpus
v_size = len(vocab)
m_size = f"{v_size} x {v_size}"

# Chạy vòng lặp để thu thập số liệu cho từng kích thước cửa sổ
for w_size in [1, 2, 5]:
    co_matrix = build_cooccurrence_matrix(text_corpus, vocab, window_size=w_size)
    stats = compute_matrix_stats(co_matrix)

    sim_synonym = word_similarity("doctor", "physician", co_matrix, vocab)
    sim_related = word_similarity("doctor", "hospital", co_matrix, vocab)

    comparison_results.append({
        "Window Size": w_size,
        "Vocabulary Size": v_size,
        "Matrix Size": m_size,
        "Non-zero Values": stats['non_zero'],
        "Sparsity (%)": round(stats['sparsity_percent'], 2),
        "Sim (doctor, physician)": round(sim_synonym, 4),
        "Sim (doctor, hospital)": round(sim_related, 4)
    })

# Tạo DataFrame và in ra bảng so sánh tổng quát
df_comparison = pd.DataFrame(comparison_results)
print("Compare Table \n")
print(df_comparison.to_string(index=False))

Compare Table 

 Window Size  Vocabulary Size Matrix Size  Non-zero Values  Sparsity (%)  Sim (doctor, physician)  Sim (doctor, hospital)
           1               89     89 x 89              216         97.27                   0.5000                  0.7071
           2               89     89 x 89              390         95.08                   0.6667                  0.4082
           5               89     89 x 89              654         91.74                   0.8750                  0.9487


#Experiment 2 — Word2Vec

In [16]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 43.1 MB/s eta 0:00:00


In [22]:
from gensim.models import Word2Vec
w2v_model = Word2Vec(
    sentences=text_corpus,
    vector_size=10,
    window=5,
    min_count=1,
    epochs=10,
    sg=1
)
sim = w2v_model.wv.similarity('doctor', 'physician')
print(f"Cosine Similarity (doctor, physician): {sim:.4f}")

Cosine Similarity (doctor, physician): 0.3559



1. **Corpus:** Tập dữ liệu tự dựng mini-corpus gồm 20 câu tiếng Anh, bao phủ các chủ đề khác nhau như động vật, y tế, nghề nghiệp và khoa học.

2. **Vocabulary:** 89 từ, được trích xuất tự động từ corpus sau khi lọc qua `min_count`.

3. **Embedding dimension:** `vector_size = 10`. Mỗi từ được biểu diễn dưới dạng một vector thực có 10 chiều.

4. **Window:** `window = 5`. Mô hình sử dụng tối đa 5 từ bên trái và 5 từ bên phải của từ trung tâm làm ngữ cảnh.

5. **Min count:** `min_count = 1`. Giữ lại toàn bộ các từ xuất hiện ít nhất 1 lần trong ngữ liệu. Với các tập dữ liệu lớn trong thực tế, giá trị này thường được đặt cao hơn để loại bỏ các từ hiếm hoặc nhiễu.

6. **Epochs:** `epochs = 10`. Mô hình duyệt qua toàn bộ corpus 10 lần để cập nhật và tinh chỉnh các trọng số embedding.

7. **Training objective:** Skip-gram với `sg = 1`. Mô hình sử dụng một từ trung tâm để dự đoán các từ xuất hiện trong ngữ cảnh xung quanh nó.


In [23]:
top_5_doctor = w2v_model.wv.most_similar("doctor", topn=5)

print("Top 5 từ tương đồng với 'doctor':")
for word, score in top_5_doctor:
    print(f"- {word}: {score:.4f}")

Top 5 từ tương đồng với 'doctor':
- reads: 0.7407
- bread: 0.7196
- subset: 0.6264
- sick: 0.4674
- living: 0.4661



## Giải thích kết quả Top 5 của từ `doctor`

### 1. Đối với từ `sick` với điểm tương đồng 0.4674

- **Bằng chứng từ corpus:** Từ `doctor` chỉ xuất hiện trong một câu: *"the doctor treats sick patients in the hospital"*.
- **Giải thích:** Từ `sick` nằm gần `doctor` và hoàn toàn nằm trong phạm vi cửa sổ ngữ cảnh `window = 5`. Trong quá trình huấn luyện Skip-gram, cặp từ này được sử dụng để cập nhật các vector embedding. Do đó, vector của `doctor` và `sick` có xu hướng trở nên gần nhau trong không gian vector.

### 2. Đối với các từ `reads`, `bread`, `subset`, `living`

Các từ này có điểm tương đồng cao mặc dù không có mối quan hệ ngữ nghĩa rõ ràng với `doctor`.

- **Bằng chứng từ corpus:** Trong tập 20 câu, các từ `doctor`, `reads`, `bread`, `subset` và `living` đều chỉ xuất hiện một lần.
- **Bằng chứng từ ngữ cảnh:** Các từ này xuất hiện trong những câu có cấu trúc tương tự và thường nằm gần các từ xuất hiện phổ biến như `the`, `a` và `in`. Ví dụ:
  - *"the doctor treats..."*
  - *"the brilliant student reads..."*
  - *"makes fresh bread in the..."*
  - *"is a subset of..."*

- **Giải thích:** Điểm tương đồng cao trong trường hợp này chủ yếu đến từ kích thước corpus quá nhỏ và dữ liệu thưa. Khi `min_count = 1`, các từ chỉ xuất hiện một lần vẫn được giữ lại, nhưng mô hình có rất ít thông tin để học được đặc trưng ổn định cho chúng. Các từ hiếm có thể nhận được những cập nhật embedding tương tự do xuất hiện trong những ngữ cảnh có các từ phổ biến giống nhau.

Vì vậy, các kết quả này không nên được xem là bằng chứng cho thấy những từ trên có quan hệ ngữ nghĩa với `doctor`. Chúng phản ánh hạn chế của việc huấn luyện Word2Vec trên một corpus rất nhỏ, trong đó mỗi từ hiếm chỉ có một vài mẫu ngữ cảnh để học.


#Experiment 3 — Context window

In [25]:
import pandas as pd
from gensim.models import Word2Vec

# Corpus mới:
new_corpus = [
    "the experienced doctor carefully examines the sick patient every morning at the city hospital".split(),
    "the skilled physician carefully examines the sick patient every morning at the local clinic".split(),
    "the friendly doctor always greets the nurses before starting rounds in the general hospital".split(),
    "the chief physician always greets the nurses before starting rounds in the private clinic".split(),
    "a good doctor must study for many years to work in a modern hospital".split(),
    "a good physician must study for many years to work in a modern healthcare facility".split(),

    "the cute fluffy cat plays with a small ball of yarn in the yard".split(),
    "the cute fluffy dog plays with a small rubber bone in the yard".split(),
    "my lazy cat sleeps all day on the warm sofa in the living room".split(),
    "my lazy dog sleeps all day on the warm rug in the living room".split(),
    "the hungry cat quickly eats the fresh food from the small round bowl".split(),
    "the hungry dog quickly eats the fresh food from the large metal bowl".split()
]

results = []
word_pairs = [("doctor", "physician"), ("doctor", "hospital"), ("cat", "dog")]

for w in [2, 5, 10]:
    model = Word2Vec(sentences=new_corpus, vector_size=100, window=w, min_count=1, epochs=50, sg=1)

    sim_scores = {"Window Size": f"Window = {w}"}
    for w1, w2 in word_pairs:
        if w1 in model.wv.key_to_index and w2 in model.wv.key_to_index:
            sim_scores[f"{w1} - {w2}"] = round(model.wv.similarity(w1, w2), 4)
        else:
            sim_scores[f"{w1} - {w2}"] = "N/A"

    results.append(sim_scores)

# Hiển thị bảng
df_results = pd.DataFrame(results)
print("Compare table for CONTEXT WINDOW\n")
print(df_results.to_string(index=False))

Compare table for CONTEXT WINDOW

Window Size  doctor - physician  doctor - hospital  cat - dog
 Window = 2              0.5049             0.3430     0.4591
 Window = 5              0.9488             0.9300     0.9510
Window = 10              0.9791             0.9749     0.9825


### Ảnh hưởng của kích thước Window

**Kết luận:** Không. Việc tăng `Window` quá lớn trên một corpus nhỏ không giúp mô hình tốt hơn mà làm giảm khả năng phân biệt ngữ nghĩa giữa các từ.

- **Phân tích từ thực nghiệm:** Với `Window = 2`, mô hình vẫn giữ được sự phân hóa giữa các từ. Cặp `doctor - physician` có điểm tương đồng `0.5049`, cao hơn rõ rệt so với `doctor - hospital` với `0.3430`.

- **Ảnh hưởng của Window quá lớn:** Khi tăng lên `Window = 5` và `Window = 10`, điểm tương đồng của các cặp từ đều tăng mạnh, lần lượt lên khoảng `0.93` và `0.97`. Với độ dài trung bình của câu trong corpus chỉ khoảng 14 từ, `Window = 10` gần như bao phủ toàn bộ câu.

- **Nguyên nhân:** Khi cửa sổ quá lớn, mô hình xem nhiều từ trong cùng một câu là có ngữ cảnh tương tự nhau, kể cả khi chúng không có quan hệ ngữ nghĩa rõ ràng. Điều này làm các vector embedding trở nên quá giống nhau và giảm khả năng phân biệt giữa các từ.

Vì vậy, với corpus nhỏ, `Window = 2` cho khả năng phân hóa tốt hơn so với các giá trị lớn như `5` hoặc `10`.

#Experiment 4 — Embedding dimension

In [29]:
import gzip
import json

file_path = 'c4-train.00000-of-01024-30K.json.gz'
corpus_30k = []
MAX_DOCS = 5000  # Bạn có thể chỉnh xuống 3000 hoặc lên 10000 tùy ý

with gzip.open(file_path, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= MAX_DOCS:
            break

        data = json.loads(line)
        text = data.get('text', '')
        tokens = text.lower().split()

        if tokens:
            corpus_30k.append(tokens)

print(f"Đã nạp nhanh {len(corpus_30k)} văn bản để test!")

Đã nạp nhanh 5000 văn bản để test!


In [31]:
import time
import numpy as np
import pandas as pd
from gensim.models import Word2Vec



results = []
dimensions_to_test = [50, 100, 300]

for dim in dimensions_to_test:
    print(f"Đang train model với dimension = {dim}...")

    start_time = time.time()
    model = Word2Vec(sentences=corpus_30k, vector_size=dim, window=5, min_count=5, epochs=10, workers=4)
    train_time = time.time() - start_time

    model_size_mb = model.wv.vectors.nbytes / (1024 * 1024)

    # 3. Đánh giá Similarity (Dùng đồ công nghệ: iphone và ipad)
    try:
        sim_score = model.wv.similarity('iphone', 'ipad')
    except KeyError:
        sim_score = 0.0

    # 4. Đánh giá Analogy (Thủ đô: paris - france + japan = tokyo)
    try:
        analogy_result = model.wv.most_similar(positive=['paris', 'japan'], negative=['france'], topn=1)[0][0]
    except KeyError:
        analogy_result = "N/A"

    results.append({
        "Dimension": dim,
        "Train Time (sec)": round(train_time, 2),
        "Model Size (MB)": round(model_size_mb, 2),
        "Sim (iphone, ipad)": round(sim_score, 4),
        "Analogy (paris-france+japan)": analogy_result
    })

df_results = pd.DataFrame(results)
print("\n--- KẾT QUẢ SO SÁNH EXPERIMENT 4 ---")
print(df_results.to_string(index=False))

Đang train model với dimension = 50...
Đang train model với dimension = 100...
Đang train model với dimension = 300...

--- KẾT QUẢ SO SÁNH EXPERIMENT 4 ---
 Dimension  Train Time (sec)  Model Size (MB)  Sim (iphone, ipad) Analogy (paris-france+japan)
        50             32.76             4.82              0.6603                     two-year
       100             37.28             9.64              0.7547                         2006
       300             45.74            28.93              0.7341                      academy


#Evaluation — Word similarity

In [32]:
pairs_21 = [
    ("doctor", "physician"),
    ("car", "automobile"),
    ("king", "queen"),
    ("cat", "dog"),
    ("computer", "banana")
]

eval_scores = []
for w1, w2 in pairs_21:
    try:
        score = model.wv.similarity(w1, w2)
    except KeyError:
        score = 0.0
    eval_scores.append((f"{w1} - {w2}", round(score, 4)))

# Xếp hạng giảm dần để đưa ra Quantitative result
eval_scores.sort(key=lambda x: x[1], reverse=True)

df_eval = pd.DataFrame(eval_scores, columns=["Word Pair", "Similarity Score"])
print("\n--- BẢNG XẾP HẠNG ĐỘ TƯƠNG ĐỒNG  ---")
print(df_eval.to_string(index=False))


--- BẢNG XẾP HẠNG ĐỘ TƯƠNG ĐỒNG (MỤC 21) ---
         Word Pair  Similarity Score
      king - queen            0.7379
         cat - dog            0.5778
doctor - physician            0.5077
  car - automobile            0.2450
 computer - banana            0.1011


In [34]:
df_eval.to_csv('results.csv', index=False)

### Phân tích kết quả xếp hạng các cặp từ

**1. Xếp hạng**

Dựa trên điểm số Cosine Similarity, mô hình tự động xếp hạng các cặp từ theo thứ tự giảm dần:

`king - queen` (0.7379) > `cat - dog` (0.5778) > `doctor - physician` (0.5077) > `car - automobile` (0.2450) > `computer - banana` (0.1011)

**2. So sánh với trực giác**

Phần lớn thứ hạng phù hợp với trực giác ngữ nghĩa:

- Các cặp từ thuộc cùng một phạm trù như `king - queen` và `cat - dog` có điểm tương đồng cao hơn.
- Cặp `computer - banana` không có mối quan hệ ngữ nghĩa rõ ràng nên có điểm thấp nhất, chỉ `0.1011`.
- Kết quả cho thấy mô hình đã phần nào học được sự khác biệt về ngữ cảnh giữa các nhóm từ.

**3. Trường hợp bất ngờ**

Điểm đáng chú ý nhất là cặp `car - automobile` với điểm tương đồng chỉ `0.2450`.

- **Theo trực giác:** Đây là hai từ gần như đồng nghĩa nên đáng lẽ phải có mức tương đồng cao.
- **Nguyên nhân:** Kết quả chịu ảnh hưởng lớn từ dữ liệu huấn luyện. `car` là từ phổ biến và xuất hiện trong nhiều loại ngữ cảnh, trong khi `automobile` ít phổ biến hơn và thường xuất hiện trong những ngữ cảnh trang trọng hoặc chuyên biệt. Do hai từ có phân bố ngữ cảnh khác nhau, Word2Vec có thể học được hai vector không đủ gần nhau, dù về mặt từ điển chúng có ý nghĩa tương đương.

Điều này cho thấy Cosine Similarity của Word2Vec không chỉ phụ thuộc vào quan hệ đồng nghĩa giữa các từ mà còn phụ thuộc rất lớn vào tần suất xuất hiện và sự đa dạng của ngữ cảnh trong corpus.

In [35]:
# Thực hiện phép toán đại số vector: king - man + woman
try:
    analogy_result = model.wv.most_similar(positive=['king', 'woman'], negative=['man'], topn=3)
    print("king - man + woman ≈")
    for word, score in analogy_result:
        print(f"  -> {word} (score: {score:.4f})")
except KeyError as e:
    print(f"Lỗi: {e} chưa có trong từ vựng.")

king - man + woman ≈
  -> opera (score: 0.8080)
  -> dayton (score: 0.7785)
  -> salem, (score: 0.7629)


### Phân tích phép toán Vector Arithmetic

**1. Tại sao phép toán lại trả về `opera` và `dayton`?**

- **Dữ liệu thưa:** Mô hình chỉ được huấn luyện trên 5.000 dòng được lấy từ tập C4 nên lượng dữ liệu còn hạn chế. Các mối quan hệ giữa `king`, `queen`, `man` và `woman` có thể chưa được học đủ rõ.
- **Ngữ cảnh không đồng nhất:** Các từ như `king` có thể xuất hiện trong nhiều ngữ cảnh khác nhau như `Burger King`, `Stephen King` hoặc `King size bed`, thay vì chỉ xuất hiện trong ngữ cảnh hoàng gia. Vì vậy, vector của `king` không nhất thiết đại diện riêng cho khái niệm vua.
- Khi thực hiện phép toán `king - man + woman`, vector kết quả có thể không nằm gần `queen`. Hàm `most_similar` chỉ tìm những từ có vector gần với vị trí kết quả nhất, nên có thể trả về những từ như `opera` hoặc `dayton` dù chúng không có quan hệ rõ ràng với phép toán ban đầu.

**2. Tại sao Vector Arithmetic có thể biểu diễn quan hệ khi được huấn luyện trên đủ dữ liệu?**

Khả năng thực hiện các phép toán trên vector xuất phát từ **Distributional Hypothesis**: những từ xuất hiện trong các ngữ cảnh tương tự thường có xu hướng có biểu diễn vector gần nhau.

Khi được huấn luyện trên một corpus đủ lớn, Word2Vec có thể học được những mẫu quan hệ thường xuyên xuất hiện. Chẳng hạn, sự khác biệt về ngữ cảnh giữa `man` và `woman` có thể có hướng tương tự với sự khác biệt giữa `king` và `queen`.

Do đó, phép toán:

`king - man + woman`

có thể tạo ra một vector nằm gần `queen`.

Tuy nhiên, đây không phải là phép suy luận logic theo nghĩa thông thường. Mối quan hệ này xuất hiện vì mô hình đã học được các quy luật phân bố trong không gian vector từ một lượng lớn dữ liệu.

**3. Hạn chế về khả năng hiểu ngữ nghĩa**

Kết quả `opera` cho thấy Word2Vec không thực hiện suy luận dựa trên kiến thức hay hiểu biết như con người. Mô hình chỉ học các quan hệ thống kê giữa từ và ngữ cảnh rồi biểu diễn chúng dưới dạng các vector số.

Khi dữ liệu huấn luyện quá ít hoặc chứa nhiều ngữ cảnh khác nhau, các vector có thể không phản ánh đúng quan hệ mà chúng ta mong đợi. Trong trường hợp này, phép toán vẫn được thực hiện chính xác về mặt toán học nhưng kết quả có thể không có ý nghĩa đối với con người.

Điều này cho thấy Word2Vec phụ thuộc rất lớn vào dữ liệu huấn luyện và **không có khả năng tự nhận biết rằng một kết quả là hợp lý hay vô lý**.

In [37]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. Tạo tập documents mẫu (Corpus tìm kiếm)
# Cố tình KHÔNG chứa chính xác các từ "medical" hay "treatment" trong các câu y tế
documents = [
    "the hospital provides clinical therapy and cares for the sick",
    "doctors prescribe medication to help patients recover from diseases",
    "the new electric automobile has an amazing battery life",
    "athletes undergo intensive training for the football championship",
    "machine learning algorithms require large amounts of data"
]

# 2. Hàm biểu diễn câu/tài liệu thành 1 vector duy nhất (Sentence/Document Embedding)
def get_document_vector(doc_text, w2v_model):
    words = doc_text.lower().split()
    valid_words = [w for w in words if w in w2v_model.wv.key_to_index]

    # Nếu document không có từ nào nằm trong vocab, trả về vector 0
    if not valid_words:
        return np.zeros(w2v_model.vector_size)

    # Kỹ thuật Mean Pooling: Trung bình cộng vector của tất cả các từ trong câu
    word_vectors = np.array([w2v_model.wv[w] for w in valid_words])
    return np.mean(word_vectors, axis=0)

# 3. Hàm Semantic Search
def semantic_search(query, docs, w2v_model, top_k=2):
    # Bước phụ: In ra các từ có ngữ nghĩa gần query để sinh viên giải thích (Query Expansion)
    query_words = query.lower().split()
    expanded_words = []
    for w in query_words:
        if w in w2v_model.wv.key_to_index:
            sim_words = [item[0] for item in w2v_model.wv.most_similar(w, topn=4)]
            expanded_words.extend(sim_words)

    if expanded_words:
        print(f"--- Mạng nơ-ron liên kết Query với các khái niệm: {', '.join(set(expanded_words))} ---\n")

    # Vector hóa Query và toàn bộ Documents
    query_vec = get_document_vector(query, w2v_model).reshape(1, -1)
    doc_vecs = np.array([get_document_vector(doc, w2v_model) for doc in docs])

    # Tính Cosine Similarity
    sim_scores = cosine_similarity(query_vec, doc_vecs)[0]

    # Xếp hạng kết quả (Ranking)
    ranked_indices = np.argsort(sim_scores)[::-1]

    print(f"KẾT QUẢ TÌM KIẾM CHO QUERY: '{query}'")
    for i in range(top_k):
        idx = ranked_indices[i]
        print(f"Top {i+1} (Score: {sim_scores[idx]:.4f}) -> {docs[idx]}")

# 4. Chạy thực nghiệm
semantic_search("medical treatment", documents, model)
semantic_search("patient", documents, model)

--- Mạng nơ-ron liên kết Query với các khái niệm: mechanical, diagnosis, enforcement, risk, organizations,, reduction, clinical, regulatory ---

KẾT QUẢ TÌM KIẾM CHO QUERY: 'medical treatment'
Top 1 (Score: 0.5428) -> machine learning algorithms require large amounts of data
Top 2 (Score: 0.4166) -> the hospital provides clinical therapy and cares for the sick
--- Mạng nơ-ron liên kết Query với các khái niệm: diagnosis, patients, treatment, stress ---

KẾT QUẢ TÌM KIẾM CHO QUERY: 'patient'
Top 1 (Score: 0.4993) -> machine learning algorithms require large amounts of data
Top 2 (Score: 0.3884) -> doctors prescribe medication to help patients recover from diseases


In [40]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity



def compare_word_pair(w1, w2, w2v_model):
    """Tính độ tương đồng Cosine giữa 2 vector từ đơn"""
    w1, w2 = w1.lower(), w2.lower()
    if w1 in w2v_model.wv.key_to_index and w2 in w2v_model.wv.key_to_index:
        v1 = w2v_model.wv[w1].reshape(1, -1)
        v2 = w2v_model.wv[w2].reshape(1, -1)
        sim = cosine_similarity(v1, v2)[0][0]
        print(f"Cosine Similarity ('{w1}', '{w2}') = {sim:.4f}")
        return sim
    else:
        print(f"Từ '{w1}' hoặc '{w2}' không có trong từ vựng (OOV).")
        return None

def analyze_word_by_word_matrix(query, doc, w2v_model):
    """Lập bảng so sánh Cosine Similarity giữa từng từ trong Query và từng từ trong Document"""
    q_words = [w for w in query.lower().split() if w in w2v_model.wv.key_to_index]
    d_words = [w for w in doc.lower().split() if w in w2v_model.wv.key_to_index]

    print(f"\n--- MA TRẬN SO SÁNH TỪNG TỪ: Query ('{query}') VS Document ---")
    print(f"Document: '{doc}'")
    print(f"{'Query Word':<15} | {'Doc Word':<15} | {'Cosine Similarity':<18}")
    print("-" * 52)

    for qw in q_words:
        for dw in d_words:
            v_q = w2v_model.wv[qw].reshape(1, -1)
            v_d = w2v_model.wv[dw].reshape(1, -1)
            sim = cosine_similarity(v_q, v_d)[0][0]
            print(f"{qw:<15} | {dw:<15} | {sim:.4f}")

# --- Ví dụ cách gọi chạy thử nghiệm từng từ ---
compare_word_pair("doctor", "nurse", model)
compare_word_pair("medical", "clinical", model)
compare_word_pair("medical", "mechanical", model)
analyze_word_by_word_matrix("medical treatment", documents[0], model)

Cosine Similarity ('doctor', 'nurse') = 0.5000
Cosine Similarity ('medical', 'clinical') = 0.8067
Cosine Similarity ('medical', 'mechanical') = 0.5495

--- MA TRẬN SO SÁNH TỪNG TỪ: Query ('medical treatment') VS Document ---
Document: 'the hospital provides clinical therapy and cares for the sick'
Query Word      | Doc Word        | Cosine Similarity 
----------------------------------------------------
medical         | the             | -0.0681
medical         | hospital        | 0.6014
medical         | provides        | 0.3180
medical         | clinical        | 0.8067
medical         | therapy         | 0.5964
medical         | and             | 0.1298
medical         | cares           | 0.1224
medical         | for             | 0.0602
medical         | the             | -0.0681
medical         | sick            | 0.0086
treatment       | the             | 0.0282
treatment       | hospital        | 0.1683
treatment       | provides        | 0.2750
treatment       | clinical      